In [1]:
#세션 기반 퍼널, 전환율, 이탈률

In [ ]:
# 02. 세션 기반 구매 퍼널 분석
# 사용자 세션을 기준으로 view → cart → purchase 전환 과정과 단계별 이탈을 분석

In [134]:
# 퍼널 분석에 사용할 라이브러리와 데이터 경로 설정

import duckdb
from pathlib import Path

parquet_path = r"../data/processed/20*.parquet"

funnel_start_date = "2019-12-01"
funnel_end_date = "2020-05-01"

anomaly_dates = [
    "2020-01-01",
    "2020-01-02",
    "2020-01-03",
    "2020-02-27",
    "2020-04-20",
    "2020-04-21"
]

anomaly_dates_sql = ", ".join(f"'{date}'" for date in anomaly_dates)

sessionized_events_parquet = r"../data/processed/sessionized_events.parquet"

In [136]:
# 메인 퍼널 분석 대상 기간과 이벤트 수 확인

duckdb.sql(f"""
    SELECT
        MIN(event_time) AS min_event_time,
        MAX(event_time) AS max_event_time,
        COUNT(*) AS event_count

    FROM read_parquet('{parquet_path}')

    WHERE event_time >= '{funnel_start_date}'
      AND event_time < '{funnel_end_date}'
      AND CAST(event_time AS DATE) NOT IN ({anomaly_dates_sql})
""").show()

┌─────────────────────┬─────────────────────┬─────────────┐
│   min_event_time    │   max_event_time    │ event_count │
│      timestamp      │      timestamp      │    int64    │
├─────────────────────┼─────────────────────┼─────────────┤
│ 2019-12-01 00:00:00 │ 2020-04-30 23:59:59 │   291603448 │
└─────────────────────┴─────────────────────┴─────────────┘



In [137]:
# 재구성된 분석 세션×상품별 최초 View 시점 생성

first_view_parquet = r"../data/processed/first_view.parquet"

Path(first_view_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            user_id,
            user_session,
            analysis_session_number,
            product_id,

            MIN(event_time) AS first_view_time

        FROM read_parquet('{sessionized_events_parquet}')

        WHERE event_type = 'view'

        GROUP BY
            user_id,
            user_session,
            analysis_session_number,
            product_id

    )
    TO '{first_view_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [139]:
# 같은 분석 세션×상품에서 최초 View 이후 발생한 첫 Cart 시점 생성

first_cart_parquet = r"../data/processed/first_cart_after_view.parquet"

Path(first_cart_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            v.user_id,
            v.user_session,
            v.analysis_session_number,
            v.product_id,
            v.first_view_time,

            MIN(e.event_time) AS first_cart_after_view

        FROM read_parquet('{first_view_parquet}') v

        JOIN read_parquet('{sessionized_events_parquet}') e
            ON v.user_id = e.user_id
           AND v.user_session = e.user_session
           AND v.analysis_session_number = e.analysis_session_number
           AND v.product_id = e.product_id

        WHERE e.event_type = 'cart'
          AND e.event_time >= v.first_view_time

        GROUP BY
            v.user_id,
            v.user_session,
            v.analysis_session_number,
            v.product_id,
            v.first_view_time

    )
    TO '{first_cart_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [140]:
# View 이후 Cart까지 도달한 세션×상품 수와 시간 범위 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS carted_session_product_count,
        MIN(first_view_time) AS min_first_view,
        MAX(first_view_time) AS max_first_view,
        MIN(first_cart_after_view) AS min_cart,
        MAX(first_cart_after_view) AS max_cart

    FROM read_parquet('{first_cart_parquet}')
""").show()

┌──────────────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│ carted_session_product_count │   min_first_view    │   max_first_view    │      min_cart       │      max_cart       │
│            int64             │      timestamp      │      timestamp      │      timestamp      │      timestamp      │
├──────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│                      9200399 │ 2019-12-01 00:00:06 │ 2020-04-30 23:59:00 │ 2019-12-01 00:00:30 │ 2020-04-30 23:59:55 │
└──────────────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┘



In [141]:
# Cart가 View보다 먼저 발생한 잘못된 행이 있는지 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS invalid_order_count

    FROM read_parquet('{first_cart_parquet}')

    WHERE first_cart_after_view < first_view_time
""").show()

┌─────────────────────┐
│ invalid_order_count │
│        int64        │
├─────────────────────┤
│                   0 │
└─────────────────────┘



In [142]:
# 같은 분석 세션×상품에서 Cart 이후 24시간 이내 발생한 첫 Purchase 시점 생성

sequential_funnel_parquet = r"../data/processed/session_product_sequential_funnel.parquet"

Path(sequential_funnel_parquet).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        SELECT
            c.user_id,
            c.user_session,
            c.analysis_session_number,
            c.product_id,
            c.first_view_time,
            c.first_cart_after_view,

            MIN(e.event_time) AS first_purchase_after_cart

        FROM read_parquet('{first_cart_parquet}') c

        JOIN read_parquet('{sessionized_events_parquet}') e
            ON c.user_id = e.user_id
           AND c.user_session = e.user_session
           AND c.analysis_session_number = e.analysis_session_number
           AND c.product_id = e.product_id

        WHERE e.event_type = 'purchase'
          AND e.event_time >= c.first_cart_after_view
          AND e.event_time < c.first_cart_after_view + INTERVAL '24 hours'

        GROUP BY
            c.user_id,
            c.user_session,
            c.analysis_session_number,
            c.product_id,
            c.first_view_time,
            c.first_cart_after_view

    )
    TO '{sequential_funnel_parquet}'
    (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [143]:
# View → Cart → Purchase까지 완료한 세션×상품 수와 시간 범위 확인

duckdb.sql(f"""
    SELECT
        COUNT(*) AS purchased_session_product_count,

        MIN(first_view_time) AS min_first_view,
        MAX(first_view_time) AS max_first_view,

        MIN(first_cart_after_view) AS min_cart,
        MAX(first_cart_after_view) AS max_cart,

        MIN(first_purchase_after_cart) AS min_purchase,
        MAX(first_purchase_after_cart) AS max_purchase

    FROM read_parquet('{sequential_funnel_parquet}')
""").show()

┌─────────────────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┬─────────────────────┐
│ purchased_session_product_count │   min_first_view    │   max_first_view    │      min_cart       │      max_cart       │    min_purchase     │    max_purchase     │
│              int64              │      timestamp      │      timestamp      │      timestamp      │      timestamp      │      timestamp      │      timestamp      │
├─────────────────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┼─────────────────────┤
│                         4379438 │ 2019-12-01 00:00:17 │ 2020-04-30 23:59:00 │ 2019-12-01 00:00:30 │ 2020-04-30 23:59:11 │ 2019-12-01 00:01:54 │ 2020-04-30 23:59:36 │
└─────────────────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴─────────────────────┴───────────────

In [144]:
# View → Cart → Purchase 순서가 깨진 행이 있는지 검증

duckdb.sql(f"""
    SELECT
        COUNT(*) AS invalid_order_count

    FROM read_parquet('{sequential_funnel_parquet}')

    WHERE first_cart_after_view < first_view_time
       OR first_purchase_after_cart < first_cart_after_view
""").show()

┌─────────────────────┐
│ invalid_order_count │
│        int64        │
├─────────────────────┤
│                   0 │
└─────────────────────┘



In [145]:
# 전체 순차 퍼널 단계별 세션×상품 수와 전환율 계산

duckdb.sql(f"""
    WITH funnel_counts AS (
        SELECT
            (SELECT COUNT(*)
             FROM read_parquet('{first_view_parquet}')) AS viewed_count,

            (SELECT COUNT(*)
             FROM read_parquet('{first_cart_parquet}')) AS carted_count,

            (SELECT COUNT(*)
             FROM read_parquet('{sequential_funnel_parquet}')) AS purchased_count
    )

    SELECT
        viewed_count,
        carted_count,
        purchased_count,

        ROUND(
            carted_count * 100.0 / NULLIF(viewed_count, 0),
            2
        ) AS view_to_cart_rate,

        ROUND(
            purchased_count * 100.0 / NULLIF(carted_count, 0),
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            purchased_count * 100.0 / NULLIF(viewed_count, 0),
            2
        ) AS sequential_funnel_completion_rate

    FROM funnel_counts
""").show()

┌──────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────────┬───────────────────────────────────┐
│ viewed_count │ carted_count │ purchased_count │ view_to_cart_rate │ cart_to_purchase_rate │ sequential_funnel_completion_rate │
│    int64     │    int64     │      int64      │      double       │        double         │              double               │
├──────────────┼──────────────┼─────────────────┼───────────────────┼───────────────────────┼───────────────────────────────────┤
│    185655039 │      9200399 │         4379438 │              4.96 │                  47.6 │                              2.36 │
└──────────────┴──────────────┴─────────────────┴───────────────────┴───────────────────────┴───────────────────────────────────┘



In [146]:
# 최초 View가 발생한 월을 기준으로 월별 순차 퍼널 계산

duckdb.sql(f"""
    WITH viewed AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS viewed_count

        FROM read_parquet('{first_view_parquet}')

        GROUP BY month
    ),

    carted AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS carted_count

        FROM read_parquet('{first_cart_parquet}')

        GROUP BY month
    ),

    purchased AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS purchased_count

        FROM read_parquet('{sequential_funnel_parquet}')

        GROUP BY month
    )

    SELECT
        v.month,
        v.viewed_count,
        c.carted_count,
        p.purchased_count,

        ROUND(
            c.carted_count * 100.0 / NULLIF(v.viewed_count, 0),
            2
        ) AS view_to_cart_rate,

        ROUND(
            p.purchased_count * 100.0 / NULLIF(c.carted_count, 0),
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            p.purchased_count * 100.0 / NULLIF(v.viewed_count, 0),
            2
        ) AS sequential_funnel_completion_rate

    FROM viewed v

    LEFT JOIN carted c
        ON v.month = c.month

    LEFT JOIN purchased p
        ON v.month = p.month

    ORDER BY v.month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬──────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────────┬───────────────────────────────────┐
│  month  │ viewed_count │ carted_count │ purchased_count │ view_to_cart_rate │ cart_to_purchase_rate │ sequential_funnel_completion_rate │
│ varchar │    int64     │    int64     │      int64      │      double       │        double         │              double               │
├─────────┼──────────────┼──────────────┼─────────────────┼───────────────────┼───────────────────────┼───────────────────────────────────┤
│ 2019-12 │     43304802 │      2298442 │         1066190 │              5.31 │                 46.39 │                              2.46 │
│ 2020-01 │     32972323 │      1522277 │          731473 │              4.62 │                 48.05 │                              2.22 │
│ 2020-02 │     35055883 │      1585659 │          782127 │              4.52 │                 49.33 │                              2.23 │
│ 2020-03 │     3544

In [147]:
# 최초 View 날짜를 기준으로 일별 순차 퍼널 계산

duckdb.sql(f"""
    WITH viewed AS (
        SELECT
            CAST(first_view_time AS DATE) AS event_date,
            COUNT(*) AS viewed_count

        FROM read_parquet('{first_view_parquet}')

        GROUP BY event_date
    ),

    carted AS (
        SELECT
            CAST(first_view_time AS DATE) AS event_date,
            COUNT(*) AS carted_count

        FROM read_parquet('{first_cart_parquet}')

        GROUP BY event_date
    ),

    purchased AS (
        SELECT
            CAST(first_view_time AS DATE) AS event_date,
            COUNT(*) AS purchased_count

        FROM read_parquet('{sequential_funnel_parquet}')

        GROUP BY event_date
    )

    SELECT
        v.event_date,
        v.viewed_count,
        COALESCE(c.carted_count, 0) AS carted_count,
        COALESCE(p.purchased_count, 0) AS purchased_count,

        ROUND(
            COALESCE(c.carted_count, 0) * 100.0
            / NULLIF(v.viewed_count, 0),
            2
        ) AS view_to_cart_rate,

        ROUND(
            COALESCE(p.purchased_count, 0) * 100.0
            / NULLIF(c.carted_count, 0),
            2
        ) AS cart_to_purchase_rate,

        ROUND(
            COALESCE(p.purchased_count, 0) * 100.0
            / NULLIF(v.viewed_count, 0),
            2
        ) AS sequential_funnel_completion_rate

    FROM viewed v

    LEFT JOIN carted c
        ON v.event_date = c.event_date

    LEFT JOIN purchased p
        ON v.event_date = p.event_date

    ORDER BY v.event_date
""").show(max_rows=200)

┌────────────┬──────────────┬──────────────┬─────────────────┬───────────────────┬───────────────────────┬───────────────────────────────────┐
│ event_date │ viewed_count │ carted_count │ purchased_count │ view_to_cart_rate │ cart_to_purchase_rate │ sequential_funnel_completion_rate │
│    date    │    int64     │    int64     │      int64      │      double       │        double         │              double               │
├────────────┼──────────────┼──────────────┼─────────────────┼───────────────────┼───────────────────────┼───────────────────────────────────┤
│ 2019-12-01 │      1135777 │        57042 │           26290 │              5.02 │                 46.09 │                              2.31 │
│ 2019-12-02 │      1092657 │        55314 │           26367 │              5.06 │                 47.67 │                              2.41 │
│ 2019-12-03 │      1059823 │        48242 │           22522 │              4.55 │                 46.69 │                              2.13 │

In [148]:
#--------------------------------------------------------------------------------------------------------------------------#

In [149]:
# 월별 순차 퍼널 결과를 Tableau용 CSV로 저장

monthly_funnel_path = r"../data/marts/dashboard_monthly_funnel.csv"

duckdb.sql(f"""
    COPY (
        WITH viewed AS (
            SELECT
                STRFTIME(first_view_time, '%Y-%m') AS month,
                COUNT(*) AS viewed_count
            FROM read_parquet('{first_view_parquet}')
            GROUP BY month
        ),

        carted AS (
            SELECT
                STRFTIME(first_view_time, '%Y-%m') AS month,
                COUNT(*) AS carted_count
            FROM read_parquet('{first_cart_parquet}')
            GROUP BY month
        ),

        purchased AS (
            SELECT
                STRFTIME(first_view_time, '%Y-%m') AS month,
                COUNT(*) AS purchased_count
            FROM read_parquet('{sequential_funnel_parquet}')
            GROUP BY month
        )

        SELECT
            v.month,
            v.viewed_count,
            c.carted_count,
            p.purchased_count,

            ROUND(
                c.carted_count * 100.0 / NULLIF(v.viewed_count, 0),
                2
            ) AS view_to_cart_rate,

            ROUND(
                p.purchased_count * 100.0 / NULLIF(c.carted_count, 0),
                2
            ) AS cart_to_purchase_rate,

            ROUND(
                p.purchased_count * 100.0 / NULLIF(v.viewed_count, 0),
                2
            ) AS sequential_funnel_completion_rate

        FROM viewed v
        LEFT JOIN carted c ON v.month = c.month
        LEFT JOIN purchased p ON v.month = p.month

        ORDER BY v.month
    )
    TO '{monthly_funnel_path}'
    (HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [150]:
# 일별 순차 퍼널 결과를 Tableau용 CSV로 저장

daily_funnel_path = r"../data/marts/dashboard_daily_funnel.csv"

duckdb.sql(f"""
    COPY (
        WITH viewed AS (
            SELECT
                CAST(first_view_time AS DATE) AS event_date,
                COUNT(*) AS viewed_count
            FROM read_parquet('{first_view_parquet}')
            GROUP BY event_date
        ),

        carted AS (
            SELECT
                CAST(first_view_time AS DATE) AS event_date,
                COUNT(*) AS carted_count
            FROM read_parquet('{first_cart_parquet}')
            GROUP BY event_date
        ),

        purchased AS (
            SELECT
                CAST(first_view_time AS DATE) AS event_date,
                COUNT(*) AS purchased_count
            FROM read_parquet('{sequential_funnel_parquet}')
            GROUP BY event_date
        )

        SELECT
            v.event_date,
            v.viewed_count,
            COALESCE(c.carted_count, 0) AS carted_count,
            COALESCE(p.purchased_count, 0) AS purchased_count,

            ROUND(
                COALESCE(c.carted_count, 0) * 100.0
                / NULLIF(v.viewed_count, 0),
                2
            ) AS view_to_cart_rate,

            ROUND(
                COALESCE(p.purchased_count, 0) * 100.0
                / NULLIF(c.carted_count, 0),
                2
            ) AS cart_to_purchase_rate,

            ROUND(
                COALESCE(p.purchased_count, 0) * 100.0
                / NULLIF(v.viewed_count, 0),
                2
            ) AS sequential_funnel_completion_rate

        FROM viewed v
        LEFT JOIN carted c ON v.event_date = c.event_date
        LEFT JOIN purchased p ON v.event_date = p.event_date

        ORDER BY v.event_date
    )
    TO '{daily_funnel_path}'
    (HEADER, DELIMITER ',')
""")

In [151]:
#--------------------------------------------------------------------------------------------------------------------------#

In [153]:
# 월별 퍼널의 단계별 이탈률 계산

duckdb.sql(f"""
    WITH viewed AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS viewed_count
        FROM read_parquet('{first_view_parquet}')
        GROUP BY month
    ),

    carted AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS carted_count
        FROM read_parquet('{first_cart_parquet}')
        GROUP BY month
    ),

    purchased AS (
        SELECT
            STRFTIME(first_view_time, '%Y-%m') AS month,
            COUNT(*) AS purchased_count
        FROM read_parquet('{sequential_funnel_parquet}')
        GROUP BY month
    )

    SELECT
        v.month,

        ROUND(
            (v.viewed_count - c.carted_count) * 100.0
            / NULLIF(v.viewed_count, 0),
            2
        ) AS view_to_cart_dropoff_rate,

        ROUND(
            (c.carted_count - p.purchased_count) * 100.0
            / NULLIF(c.carted_count, 0),
            2
        ) AS cart_to_purchase_dropoff_rate

    FROM viewed v

    LEFT JOIN carted c
        ON v.month = c.month

    LEFT JOIN purchased p
        ON v.month = p.month

    ORDER BY v.month
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬───────────────────────────┬───────────────────────────────┐
│  month  │ view_to_cart_dropoff_rate │ cart_to_purchase_dropoff_rate │
│ varchar │          double           │            double             │
├─────────┼───────────────────────────┼───────────────────────────────┤
│ 2019-12 │                     94.69 │                         53.61 │
│ 2020-01 │                     95.38 │                         51.95 │
│ 2020-02 │                     95.48 │                         50.67 │
│ 2020-03 │                     94.75 │                         49.84 │
│ 2020-04 │                     95.03 │                          55.2 │
└─────────┴───────────────────────────┴───────────────────────────────┘



In [154]:
# 월별 퍼널의 최고·최저 전환율 확인

duckdb.sql(f"""
    WITH monthly_funnel AS (
        SELECT *
        FROM read_csv_auto('{monthly_funnel_path}')
    )

    SELECT
        'View → Cart' AS metric,
        ARG_MAX(month, view_to_cart_rate) AS best_month,
        MAX(view_to_cart_rate) AS best_rate,
        ARG_MIN(month, view_to_cart_rate) AS worst_month,
        MIN(view_to_cart_rate) AS worst_rate

    FROM monthly_funnel

    UNION ALL

    SELECT
        'Cart → Purchase',
        ARG_MAX(month, cart_to_purchase_rate),
        MAX(cart_to_purchase_rate),
        ARG_MIN(month, cart_to_purchase_rate),
        MIN(cart_to_purchase_rate)

    FROM monthly_funnel

    UNION ALL

    SELECT
        'Sequential Funnel Completion',
        ARG_MAX(month, sequential_funnel_completion_rate),
        MAX(sequential_funnel_completion_rate),
        ARG_MIN(month, sequential_funnel_completion_rate),
        MIN(sequential_funnel_completion_rate)

    FROM monthly_funnel
""").show()

┌──────────────────────────────┬────────────┬───────────┬─────────────┬────────────┐
│            metric            │ best_month │ best_rate │ worst_month │ worst_rate │
│           varchar            │  varchar   │  double   │   varchar   │   double   │
├──────────────────────────────┼────────────┼───────────┼─────────────┼────────────┤
│ View → Cart                  │ 2019-12    │      5.31 │ 2020-02     │       4.52 │
│ Cart → Purchase              │ 2020-03    │     50.16 │ 2020-04     │       44.8 │
│ Sequential Funnel Completion │ 2020-03    │      2.64 │ 2020-01     │       2.22 │
└──────────────────────────────┴────────────┴───────────┴─────────────┴────────────┘



In [155]:
# 전월 대비 퍼널 전환율 변화(%p) 계산

duckdb.sql(f"""
    WITH monthly_funnel AS (
        SELECT *
        FROM read_csv_auto('{monthly_funnel_path}')
    )

    SELECT
        month,

        view_to_cart_rate,
        ROUND(
            view_to_cart_rate
            - LAG(view_to_cart_rate) OVER (ORDER BY month),
            2
        ) AS view_to_cart_change_pp,

        cart_to_purchase_rate,
        ROUND(
            cart_to_purchase_rate
            - LAG(cart_to_purchase_rate) OVER (ORDER BY month),
            2
        ) AS cart_to_purchase_change_pp,

        sequential_funnel_completion_rate,
        ROUND(
            sequential_funnel_completion_rate
            - LAG(sequential_funnel_completion_rate) OVER (ORDER BY month),
            2
        ) AS completion_change_pp

    FROM monthly_funnel

    ORDER BY month
""").show()

┌─────────┬───────────────────┬────────────────────────┬───────────────────────┬────────────────────────────┬───────────────────────────────────┬──────────────────────┐
│  month  │ view_to_cart_rate │ view_to_cart_change_pp │ cart_to_purchase_rate │ cart_to_purchase_change_pp │ sequential_funnel_completion_rate │ completion_change_pp │
│ varchar │      double       │         double         │        double         │           double           │              double               │        double        │
├─────────┼───────────────────┼────────────────────────┼───────────────────────┼────────────────────────────┼───────────────────────────────────┼──────────────────────┤
│ 2019-12 │              5.31 │                   NULL │                 46.39 │                       NULL │                              2.46 │                 NULL │
│ 2020-01 │              4.62 │                  -0.69 │                 48.05 │                       1.66 │                              2.22 │          

In [157]:
#---------------------------------------------------------------------------------------------------------------------------#

In [158]:
# 월별 퍼널 전환율과 전월 대비 변화량 저장

funnel_change_path = r"../data/marts/dashboard_funnel_change.csv"

Path(funnel_change_path).unlink(missing_ok=True)

duckdb.sql(f"""
    COPY (
        WITH monthly_funnel AS (
            SELECT *
            FROM read_csv_auto('{monthly_funnel_path}')
        )

        SELECT
            month,

            view_to_cart_rate,
            ROUND(
                view_to_cart_rate
                - LAG(view_to_cart_rate) OVER (ORDER BY month),
                2
            ) AS view_to_cart_change_pp,

            cart_to_purchase_rate,
            ROUND(
                cart_to_purchase_rate
                - LAG(cart_to_purchase_rate) OVER (ORDER BY month),
                2
            ) AS cart_to_purchase_change_pp,

            sequential_funnel_completion_rate,
            ROUND(
                sequential_funnel_completion_rate
                - LAG(sequential_funnel_completion_rate) OVER (ORDER BY month),
                2
            ) AS completion_change_pp

        FROM monthly_funnel

        ORDER BY month
    )
    TO '{funnel_change_path}'
    (HEADER, DELIMITER ',')
""")

In [159]:
#--------------------------------------------------------------------------------------------------------------------------#

In [127]:
## 분석 결과 요약

- 메인 퍼널 분석 기간은 데이터 품질 검증 결과를 반영하여 2019-12 ~ 2020-04로 설정하였다.
- 퍼널은 세션×상품 단위에서 View → 이후 Cart → 이후 Purchase 순서를 만족하는 순차 퍼널로 정의하였다.
- View→Cart 전환율은 월별 약 4.55~5.38%로, 가장 큰 이탈이 발생하는 단계였다.
- Cart→Purchase 전환율은 약 45.00~50.24% 범위였다.
- 순차 3단계 퍼널 완료율은 2020년 3월이 2.66%로 가장 높았고, 2020년 1월이 2.24%로 가장 낮았다.
- 2020년 4월에는 Cart→Purchase 전환율이 전월 대비 5.24%p 감소하면서 전체 3단계 완료율도 0.40%p 하락하였다.
- 다만 로그 데이터만으로 이러한 월별 변화의 원인을 특정할 수 없으므로 프로모션, 가격, 상품 구성 등의 원인으로 단정하지 않는다.

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (4130274521.py, line 3)